In [3]:
import os
import re
import json
import time
import hashlib
from dataclasses import dataclass
from typing import Dict, List, Any, Iterable, Optional

import requests
from tqdm import tqdm

# LangChain (lightweight local RAG ingestion)
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [4]:
# ----------------------------
# Config
# ----------------------------
IRIS_API_BASE = "https://iris.who.int/server/api"
SEARCH_ENDPOINT = f"{IRIS_API_BASE}/discover/search/objects"  # DSpace discovery search
DEFAULT_QUERY = "COVID-19 living guideline"

USER_AGENT = "toy-rag-who-iris/1.0 (contact: you@example.com)"  # change to something real
TIMEOUT = 60
SLEEP_SECS = 0.2  # be polite
PAGE_SIZE = 50    # keep modest; API supports up to 100 in examples

DOWNLOAD_DIR = "./data/who_iris_pdfs"
CHROMA_DIR = "./data/chroma_who_iris"

In [5]:
@dataclass
class IrisItemPDF:
    item_uuid: str
    item_name: str
    bitstream_id: str
    pdf_name: str
    pdf_url: str  # .../server/api/core/bitstreams/<id>/content


def _session() -> requests.Session:
    s = requests.Session()
    s.headers.update({"User-Agent": USER_AGENT, "Accept": "application/json"})
    return s


def _safe_filename(name: str, fallback: str) -> str:
    name = name.strip()
    if not name:
        return fallback
    name = re.sub(r"[^\w\-.() ]+", "_", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name[:180]

In [6]:
def iris_search_objects(query: str, max_pages: int = 20) -> List[Dict[str, Any]]:
    """
    Calls:
      https://iris.who.int/server/api/discover/search/objects?query=<q>&scope=&page=<p>&size=<n>
    as demonstrated in public examples. :contentReference[oaicite:1]{index=1}
    """
    s = _session()

    # First page to learn total pages
    params = {"query": query, "scope": "", "page": 0, "size": PAGE_SIZE}
    r = s.get(SEARCH_ENDPOINT, params=params, timeout=TIMEOUT)
    r.raise_for_status()
    data = r.json()

    page_info = data.get("_embedded", {}).get("searchResult", {}).get("page", {})
    total_pages = int(page_info.get("totalPages", 1))
    total_pages = min(total_pages, max_pages)

    objects: List[Dict[str, Any]] = []
    for p in range(total_pages):
        params = {"query": query, "scope": "", "page": p, "size": PAGE_SIZE}
        r = s.get(SEARCH_ENDPOINT, params=params, timeout=TIMEOUT)
        r.raise_for_status()
        d = r.json()
        page_objs = (
            d.get("_embedded", {})
             .get("searchResult", {})
             .get("_embedded", {})
             .get("objects", [])
        )
        objects.extend(page_objs)
        time.sleep(SLEEP_SECS)

    return objects


def _extract_item_uuid_and_name(obj: Dict[str, Any]) -> Optional[Dict[str, str]]:
    # Pattern from IRIS discovery response (indexableObject.*) shown in examples. :contentReference[oaicite:2]{index=2}
    io = obj.get("_embedded", {}).get("indexableObject", {})
    item_uuid = io.get("uuid")
    item_name = io.get("name") or "who-iris-item"
    if not item_uuid:
        return None
    return {"item_uuid": item_uuid, "item_name": item_name}


def iris_item_bitstreams(item_uuid: str) -> List[Dict[str, Any]]:
    """
    1) list bundles for item:
        /server/api/core/items/<uuid>/bundles
    2) for each bundle, list bitstreams:
        bundle._links.bitstreams.href
    """
    s = _session()

    bundles_url = f"{IRIS_API_BASE}/core/items/{item_uuid}/bundles"
    r = s.get(bundles_url, params={"size": 9999}, timeout=TIMEOUT)
    r.raise_for_status()
    bundles = r.json().get("_embedded", {}).get("bundles", [])

    bitstreams: List[Dict[str, Any]] = []
    for b in bundles:
        href = b.get("_links", {}).get("bitstreams", {}).get("href")
        if not href:
            continue
        rr = s.get(href, params={"size": 9999}, timeout=TIMEOUT)
        rr.raise_for_status()
        bs = rr.json().get("_embedded", {}).get("bitstreams", [])
        bitstreams.extend(bs)
        time.sleep(SLEEP_SECS)

    return bitstreams


In [7]:
def find_pdfs_for_query(query: str, max_pages: int = 20) -> List[IrisItemPDF]:
    """
    Converts search objects → item UUIDs → bitstreams → PDF download URLs:
      https://iris.who.int/server/api/core/bitstreams/<id>/content  :contentReference[oaicite:3]{index=3}
    """
    objs = iris_search_objects(query=query, max_pages=max_pages)
    pdfs: List[IrisItemPDF] = []

    for obj in tqdm(objs, desc="Resolving items → PDFs"):
        meta = _extract_item_uuid_and_name(obj)
        if not meta:
            continue

        item_uuid = meta["item_uuid"]
        item_name = meta["item_name"]

        try:
            bitstreams = iris_item_bitstreams(item_uuid)
        except Exception as e:
            # Some items may not expose bitstreams; skip
            continue

        for bs in bitstreams:
            name = (bs.get("name") or "").lower()
            mime = (bs.get("mimeType") or "").lower()
            is_pdf = name.endswith(".pdf") or ("pdf" in mime)
            if not is_pdf:
                continue

            bs_id = bs.get("id")
            if not bs_id:
                continue

            pdf_url = f"{IRIS_API_BASE}/core/bitstreams/{bs_id}/content"
            pdfs.append(
                IrisItemPDF(
                    item_uuid=item_uuid,
                    item_name=item_name,
                    bitstream_id=bs_id,
                    pdf_name=bs.get("name") or f"{bs_id}.pdf",
                    pdf_url=pdf_url,
                )
            )

    # de-dupe by bitstream id
    uniq = {}
    for p in pdfs:
        uniq[p.bitstream_id] = p
    return list(uniq.values())


def download_pdfs(pdfs: Iterable[IrisItemPDF], out_dir: str) -> List[str]:
    os.makedirs(out_dir, exist_ok=True)
    s = _session()
    s.headers.update({"Accept": "application/pdf"})

    saved_paths: List[str] = []

    for p in tqdm(list(pdfs), desc="Downloading PDFs"):
        # stable prefix to avoid collisions
        prefix = hashlib.sha1(p.bitstream_id.encode("utf-8")).hexdigest()[:10]
        fname = _safe_filename(p.pdf_name, f"{prefix}.pdf")
        path = os.path.join(out_dir, f"{prefix}_{fname}")

        if os.path.exists(path) and os.path.getsize(path) > 0:
            saved_paths.append(path)
            continue

        r = s.get(p.pdf_url, timeout=TIMEOUT, stream=True)
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 128):
                if chunk:
                    f.write(chunk)

        saved_paths.append(path)
        time.sleep(SLEEP_SECS)

    return saved_paths


def ingest_into_chroma(pdf_paths: List[str], chroma_dir: str) -> None:
    """
    Minimal local ingestion:
      PDFs → pages → chunk → HF embeddings → Chroma persist
    """
    os.makedirs(chroma_dir, exist_ok=True)

    docs = []
    for p in tqdm(pdf_paths, desc="Loading PDFs"):
        loader = PyPDFLoader(p)
        docs.extend(loader.load())

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    chunks = splitter.split_documents(docs)

    embed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    vs = Chroma.from_documents(
        documents=chunks,
        embedding=embed,
        persist_directory=chroma_dir,
        collection_name="who_iris_guidelines",
    )
    vs.persist()


In [9]:
query = os.environ.get("WHO_IRIS_QUERY", DEFAULT_QUERY)
max_pages = int(os.environ.get("WHO_IRIS_MAX_PAGES", "5"))

print(f"Query: {query}")
print("Searching WHO IRIS via REST API…")
pdfs = find_pdfs_for_query(query=query, max_pages=max_pages)

print(f"Found {len(pdfs)} PDF bitstreams.")


Query: COVID-19 living guideline
Searching WHO IRIS via REST API…


Resolving items → PDFs: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 250/250 [06:51<00:00,  1.65s/it]

Found 407 PDF bitstreams.


In [10]:
# Example of direct content URLs exists on IRIS, e.g.:
# https://iris.who.int/server/api/core/bitstreams/<uuid>/content :contentReference[oaicite:4]{index=4}

pdf_paths = download_pdfs(pdfs, DOWNLOAD_DIR)

print(f"Downloaded {len(pdf_paths)} PDFs to {DOWNLOAD_DIR}")

KeyboardInterrupt: 

In [13]:
pdf_paths = DOWNLOAD_DIR + '/*'
import glob

# Find all .txt files in the current directory
files = glob.glob(pdf_paths)
print(files)

['./data/who_iris_pdfs/936a0a5aed_B09467-eng.pdf', './data/who_iris_pdfs/66c1f419f7_B09540-eng.pdf']


In [16]:
print("Ingesting into Chroma…")
ingest_into_chroma(files, CHROMA_DIR)
print(f"Done. Chroma persisted at {CHROMA_DIR}")

Ingesting into Chroma…


Loading PDFs: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.56s/it]
/home/zoro/anaconda3/envs/blogs/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 906.91it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from diffe

Done. Chroma persisted at ./data/chroma_who_iris


/tmp/ipykernel_19086/2667434605.py:105: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vs.persist()


In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vs = Chroma(persist_directory="./data/chroma_who_iris", embedding_function=embed, collection_name="who_iris_guidelines")

docs = vs.similarity_search("When should corticosteroids be used in severe COVID-19?", k=5)
for d in docs:
    print(d.metadata, "\n", d.page_content[:400], "\n---\n")